---
description: Restore structural metadata hidden in raw OCR Markdown
output-file: preprocessing.conditioning.tree_restoration.html
title: OCR tree restoration
---

In [ ]:
# | default_exp preprocessing.conditioning.tree_restoration

# OCR tree restoration

Raw layout OCR can emit the running document title as a level-two heading at the top of every page. That title is useful metadata but is not part of the document's section hierarchy. The following conditioning step moves a repeated page-first title into its layout marker.

In [ ]:
# | export
import json
import re
import unicodedata
from collections import Counter


_PAGE_FIRST_PARAGRAPH_TITLE_RE = re.compile(
    r"(?m)"
    r"^<!--[ \t]*Page[ \t]+(?P<page>\d+)[ \t]*-->[ \t]*\r?\n"
    r"(?:[ \t]*\r?\n)*"
    r"(?P<marker><!--[ \t]*layout-region\b[^\r\n]*?"
    r"(?<!\S)label=paragraph_title(?=[ \t]|-->)[^\r\n]*?-->)[ \t]*\r?\n"
    r"(?:[ \t]*\r?\n)*"
    r"(?P<heading>[ \t]{0,3}#{1,6}[ \t]+[^\r\n]+)$"
)
_ATX_HEADING_RE = re.compile(
    r"^[ \t]{0,3}#{1,6}[ \t]+(?P<content>.*?)(?:[ \t]+#+)?[ \t]*$"
)
_PARAGRAPH_TITLE_LABEL_RE = re.compile(
    r"(?<!\S)label=paragraph_title(?=[ \t]|-->)"
)
_TITLE_WHITESPACE_RE = re.compile(r"\s+")


def _heading_content(heading: str) -> str:
    match = _ATX_HEADING_RE.fullmatch(heading)
    return match.group("content").strip() if match else ""


def _normalized_title(title: str) -> str:
    normalized = unicodedata.normalize("NFKC", title)
    return _TITLE_WHITESPACE_RE.sub(" ", normalized).strip().casefold()


def _document_title_marker(marker: str, title: str) -> str:
    marker = _PARAGRAPH_TITLE_LABEL_RE.sub(
        "label=document_title", marker, count=1
    )
    prefix, separator, _ = marker.rpartition("-->")
    if not separator:
        return marker
    content = json.dumps(title, ensure_ascii=False)
    return f"{prefix.rstrip()} content={content} -->"


In [ ]:
# | export
def absorb_redundant_document_titles(
    markdown: str,
    *,
    document_title: str | None = None,
    min_occurrences: int = 2,
) -> str:
    """Move a repeated page-first document heading into its layout marker.

    A candidate must be a ``paragraph_title`` region immediately below a
    ``<!-- Page N -->`` delimiter and contain one ATX Markdown heading. If
    ``document_title`` is omitted, the most frequent candidate is absorbed
    only when it occurs at least ``min_occurrences`` times. Supplying a title
    selects exact Unicode/whitespace-normalized matches and also supports a
    one-page document.

    The heading is removed, the marker label becomes ``document_title``, and
    its text is stored as a JSON-quoted ``content`` property. Unmatched text
    is preserved exactly.
    """
    if min_occurrences < 1:
        raise ValueError("min_occurrences must be at least 1")

    candidates: list[tuple[re.Match[str], str, str]] = []
    for match in _PAGE_FIRST_PARAGRAPH_TITLE_RE.finditer(markdown):
        title = _heading_content(match.group("heading"))
        normalized = _normalized_title(title)
        if normalized:
            candidates.append((match, title, normalized))

    if not candidates:
        return markdown

    if document_title is not None:
        selected_title = _normalized_title(document_title)
        if not selected_title:
            raise ValueError("document_title must not be empty")
    else:
        counts = Counter(normalized for _, _, normalized in candidates)
        selected_title, count = counts.most_common(1)[0]
        if count < min_occurrences:
            return markdown

    parts: list[str] = []
    cursor = 0
    for match, title, normalized in candidates:
        if normalized != selected_title:
            continue
        marker_start, marker_end = match.span("marker")
        _, heading_end = match.span("heading")
        parts.append(markdown[cursor:marker_start])
        parts.append(_document_title_marker(markdown[marker_start:marker_end], title))
        cursor = heading_end
    parts.append(markdown[cursor:])
    return "".join(parts)


In [ ]:
# | hide
from fastcore.test import test_eq, test_fail


def _region(page: int, index: int, label: str, content: str) -> str:
    return (
        f"<!-- Page {page} -->\n\n"
        f"<!-- layout-region page={page} index={index} label={label} "
        "bbox=474,179,1082,246 status=completed -->\n\n"
        f"{content}\n"
    )


def test_absorb_redundant_document_titles():
    title = "新松机器人通用操作手册"
    markdown = (
        _region(3, 1, "paragraph_title", "## 安全须知")
        + _region(19, 1, "paragraph_title", f"## {title}")
        + "Body on page 19\n"
        + _region(20, 1, "paragraph_title", f"## {title}")
        + "Body on page 20\n"
    )

    conditioned = absorb_redundant_document_titles(markdown)

    test_eq(conditioned.count("label=document_title"), 2)
    test_eq(conditioned.count(f'content="{title}"'), 2)
    assert f"## {title}" not in conditioned
    assert "label=paragraph_title" in conditioned
    assert "## 安全须知" in conditioned
    assert "Body on page 19" in conditioned
    test_eq(absorb_redundant_document_titles(conditioned), conditioned)


def test_unique_title_requires_explicit_selection():
    markdown = _region(116, 1, "paragraph_title", "## Manual \"B\"")
    test_eq(absorb_redundant_document_titles(markdown), markdown)
    conditioned = absorb_redundant_document_titles(
        markdown, document_title='Manual "B"'
    )
    assert 'label=document_title' in conditioned
    expected_content = "content=" + json.dumps('Manual "B"')
    assert expected_content in conditioned
    assert "## Manual" not in conditioned


def test_non_page_first_heading_is_not_absorbed():
    markdown = (
        "<!-- Page 1 -->\n\n"
        "<!-- layout-region page=1 index=1 label=text "
        "bbox=0,0,1,1 status=completed -->\n\nBody\n\n"
        "<!-- layout-region page=1 index=2 label=paragraph_title "
        "bbox=0,2,1,3 status=completed -->\n\n## Manual\n"
    )
    test_eq(
        absorb_redundant_document_titles(markdown, document_title="Manual"),
        markdown,
    )
    test_fail(
        lambda: absorb_redundant_document_titles(markdown, min_occurrences=0),
        contains="min_occurrences",
    )


test_absorb_redundant_document_titles()
test_unique_title_requires_explicit_selection()
test_non_page_first_heading_is_not_absorbed()
